In [ ]:
!pip install tf-nightly

In [ ]:
from keras.applications import vgg16
from keras.preprocessing.image import load_img, img_to_array
from keras.models import Model
from keras.applications.imagenet_utils import preprocess_input
from PIL import Image
import os
import matplotlib.pyplot as plt
import numpy as np 
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
imgs_path = "/Users/arshi/Desktop/vgg_image-net/data/Apparel/Boys/Images/images_with_product_ids"
imgs_model_width, imgs_model_height = 224, 224
nb_closest_images = 10

In [ ]:
vgg_model = vgg16.VGG16(weights='imagenet')

feat_extractor = Model(inputs=vgg_model.input, outputs=vgg_model.get_layer("fc2").output)

feat_extractor.summary()

In [ ]:
files = [imgs_path + "/" + x for x in os.listdir(imgs_path) if "jpg" in x]
print("number of images:",len(files))

In [ ]:
original = load_img(files[4], target_size=(imgs_model_width, imgs_model_height))
plt.imshow(original)
plt.show()
print("image loaded successfully!")

In [ ]:
numpy_image = img_to_array(original)

image_batch = np.expand_dims(numpy_image, axis=0)
print('image batch size', image_batch.shape)

In [ ]:
processed_image = preprocess_input(image_batch.copy())
img_features = feat_extractor.predict(processed_image)

print("number of image features:",img_features.size)
img_features

In [ ]:
importedImages = []

for f in files:
    filename = f
    original = load_img(filename, target_size=(224, 224))
    numpy_image = img_to_array(original)
    image_batch = np.expand_dims(numpy_image, axis=0)
    
    importedImages.append(image_batch)

In [ ]:
images = np.vstack(importedImages)
processed_imgs = preprocess_input(images.copy())
imgs_features = feat_extractor.predict(processed_imgs)

print("features successfully extracted!")
print("number of image features:",imgs_features.size)
imgs_features.shape

In [ ]:
cosSimilarities = cosine_similarity(imgs_features)
cos_similarities_df = pd.DataFrame(cosSimilarities, columns=files, index=files)

In [ ]:
def retrieve_most_similar_products(given_img):
    print("original product:")
    original = load_img(given_img, target_size=(imgs_model_width, imgs_model_height))
    plt.imshow(original)
    plt.show()

    print("most similar products:")
    closest_imgs = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1].index
    closest_imgs_scores = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1]

    for i in range(0,len(closest_imgs)):
        original = load_img(closest_imgs[i], target_size=(imgs_model_width, imgs_model_height))
        plt.imshow(original)
        plt.show()
        print("similarity score : ",closest_imgs_scores[i])

In [ ]:
retrieve_most_similar_products(files[45])

In [ ]:
retrieve_most_similar_products (files[90])

In [ ]:
class ImageSimilarity:
    def __init__(self, imgs_path, nb_closest_images=10):
        self.imgs_path = imgs_path
        self.nb_closest_images = nb_closest_images
        self.imgs_model_width, self.imgs_model_height = 224, 224

        self.vgg_model = vgg16.VGG16(weights='imagenet')
        self.feat_extractor = Model(inputs=self.vgg_model.input,
                                    outputs=self.vgg_model.get_layer("fc2").output)

        self.files = [imgs_path + "/" + x for x in os.listdir(imgs_path) if "jpg" in x]
        self.imported_images = self.load_images()
        self.imgs_features = self.extract_features()
        self.cos_similarities_df = self.calculate_cosine_similarity()
    
    def load_model(self, path):
        self.feat_extractor = load_model(path)

    def load_images(self):
        imported_images = []
        for f in self.files:
            original = load_img(f, target_size=(self.imgs_model_width, self.imgs_model_height))
            numpy_image = img_to_array(original)
            image_batch = np.expand_dims(numpy_image, axis=0)
            imported_images.append(image_batch)

        return np.vstack(imported_images)

    def extract_features(self):
        processed_imgs = preprocess_input(self.imported_images.copy())
        return self.feat_extractor.predict(processed_imgs)

    def calculate_cosine_similarity(self):
        cos_similarities = cosine_similarity(self.imgs_features)
        return pd.DataFrame(cos_similarities, columns=self.files, index=self.files)

    def retrieve_most_similar_products(self, given_img):
        print("Original product:")
        original = load_img(given_img, target_size=(self.imgs_model_width, self.imgs_model_height))
        plt.imshow(original)
        plt.show()

        print("Most similar products:")
        closest_imgs = self.cos_similarities_df[given_img].sort_values(ascending=False)[1:self.nb_closest_images+1].index
        closest_imgs_scores = self.cos_similarities_df[given_img].sort_values(ascending=False)[1:self.nb_closest_images+1]

        for i in range(len(closest_imgs)):
            original = load_img(closest_imgs[i], target_size=(self.imgs_model_width, self.imgs_model_height))
            plt.imshow(original)
            plt.show()
            print("Similarity score: ", closest_imgs_scores[i])
    def save_model(self, path):
        self.feat_extractor.save(path)

In [ ]:
def save_model(self, path):
    self.feat_extractor.save(path)


In [ ]:
imgs_path = "/Users/arshi/Desktop/vgg_image-net/data/Apparel/Boys/Images/images_with_product_ids"
image_similarity = ImageSimilarity(imgs_path)
model_path = "saved_model_boys"
image_similarity.save_model(model_path)


In [ ]:
from tensorflow.keras.models import load_model

feat_extractor = load_model("/Users/arshi/Desktop/vgg_image-net/inference_models/Footwear_Boys_feat_extractor.h5")


In [ ]:
from image_similarity import ImageSimilarity


image_similarity = ImageSimilarity("/Users/arshi/Desktop/vgg_image-net/data/Footwear/Men")
feat_extractor

given_img = "/Users/arshi/Desktop/vgg_image-net/data/Footwear/Men/1831.jpg"
nb_closest_images = 10
image_similarity.retrieve_most_similar_products(given_img)


In [ ]:
feat_extractor

given_img = "/Users/arshi/Desktop/vgg_image-net/data/Footwear/Men/1637.jpg"
nb_closest_images = 10
image_similarity.retrieve_most_similar_products(given_img)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from keras.applications import vgg16
from keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from keras.applications.imagenet_utils import preprocess_input

class ImageSimilarity:
    def __init__(self, imgs_path_dict, nb_closest_images=10):
        self.nb_closest_images = nb_closest_images
        self.imgs_model_width, self.imgs_model_height = 224, 224

        self.vgg_model = vgg16.VGG16(weights='imagenet')

        self.sub_categories = {}
        for category, sub_categories in imgs_path_dict.items():
            for sub_category, path in sub_categories.items():
                feat_extractor = Model(inputs=self.vgg_model.input, outputs=self.vgg_model.get_layer("fc2").output)
                files = [path + "/" + x for x in os.listdir(path) if "jpg" in x]
                imported_images = self.load_images(files)
                imgs_features = self.extract_features(feat_extractor, imported_images)
                cos_similarities_df = self.calculate_cosine_similarity(files, imgs_features)

                self.sub_categories[f"{category}_{sub_category}"] = {
                    "path": path,
                    "feat_extractor": feat_extractor,
                    "files": files,
                    "imported_images": imported_images,
                    "imgs_features": imgs_features,
                    "cos_similarities_df": cos_similarities_df
                }

    def load_images(self, files):
        imported_images = []
        for f in files:
            original = load_img(f, target_size=(self.imgs_model_width, self.imgs_model_height))
            numpy_image = img_to_array(original)
            image_batch = np.expand_dims(numpy_image, axis=0)
            imported_images.append(image_batch)

        return np.vstack(imported_images)

    def extract_features(self, feat_extractor, imported_images):
        processed_imgs = preprocess_input(imported_images.copy())
        return feat_extractor.predict(processed_imgs)

    def calculate_cosine_similarity(self, files, imgs_features):
        cos_similarities = cosine_similarity(imgs_features)
        return pd.DataFrame(cos_similarities, columns=files, index=files)

    def save_models(self, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        for sub_category, data in self.sub_categories.items():
            with open(f"{save_dir}/{sub_category}_feat_extractor.pkl", 'wb') as f:
                pickle.dump(data["feat_extractor"], f)


    def load_models(self, save_dir):
        for sub_category, data in self.sub_categories.items():
            with open(f"{save_dir}/{sub_category}_feat_extractor.pkl", 'rb') as f:
                data["feat_extractor"] = pickle.load(f)

    def load_feat_extractor(self, path):
        self.feat_extractor = load_model(path)

    def retrieve_most_similar_products(self, given_img, category, sub_category):
        key = f"{category}_{sub_category}"
        if key not in self.sub_categories:
            print(f"Category '{category}' and sub-category '{sub_category}' not found.")
            return

        data = self.sub_categories[key]

        print("Original product:")
        original = load_img(given_img, target_size=(self.imgs_model_width, self.imgs_model_height))
        plt.imshow(original)
        plt.show()

        print("Most similar products:")
        closest_imgs = data["cos_similarities_df"][given_img].sort_values(ascending=False)[1:self.nb_closest_images+1].index
        closest_imgs_scores = data["cos_similarities_df"][given_img].sort_values(ascending=False)[1:self.nb_closest_images+1]

        for i in range(len(closest_imgs)):
            original = load_img(closest_imgs[i], target_size=(self.imgs_model_width, self.imgs_model_height))
            plt.imshow(original)
            plt.show()
            print("Similarity score: ", closest_imgs_scores[i])

imgs_path_dict = {
    "Apparel": {
        "Boys": "/Users/arshi/Desktop/vgg_image-net/data/Apparel/Boys",
        "Girls": "/Users/arshi/Desktop/vgg_image-net/data/Apparel/Girls"
    },
    "Footwear": {
        "Boys": "/Users/arshi/Desktop/vgg_image-net/data/Footwear/Men",
        "Girls": "/Users/arshi/Desktop/vgg_image-net/data/Footwear/Women"
    }
}

image_similarity = ImageSimilarity(imgs_path_dict)
image_similarity.save_models("inference_models_pkl")

In [ ]:

image_similarity.load_models("/Users/arshi/Desktop/vgg_image-net/inference_models_pkl")
image_similarity.retrieve_most_similar_products("/Users/arshi/Desktop/vgg_image-net/data/Apparel/Boys/2714.jpg", "Apparel", "Boys")


In [ ]:
image_similarity.retrieve_most_similar_products("/Users/arshi/Desktop/vgg_image-net/data/Footwear/Men/1806.jpg", "Footwear", "Boys")
